In [1]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path
sys.path.append('/lab/barcheese01/smaffa/coTISja/src')

from scripts.filter_utils import *
import re

In [3]:
experiment_table, sample_df, replicate_df = load_experiment_manifest()
tisdiff_manifest = pd.read_csv('/lab/barcheese01/smaffa/coTISja/data/ribotish_tisdiff_manifest.tsv', sep='\t')
samples = experiment_table['sample'].tolist()
codon_order = ['ATG', 'ATA', 'ATC', 'ATT', 'ACG', 'AAG', 'AGG', 'GTG', 'TTG', 'CTG']

In [30]:
pairs_to_count_matrix = dict()
for _, r in tisdiff_manifest.iterrows():
    baseline_s = r['baseline']
    test_s = r['test']
    counts_file = r['export_output_file']

    counts_matrix = pd.read_csv(counts_file, sep='\t', index_col=0)

    sample_pair  = [baseline_s, test_s]
    n_reps = counts_matrix.shape[1] // 4
    assays = ['rpf', 'rna']
    
    column_names = []
    for a in assays:
        for s in sample_pair:
            for i in range(1, n_reps+1):
                column_names.append(f'{s}__rep{i}__{a}')
    counts_matrix.columns = column_names
    pairs_to_count_matrix[(test_s, baseline_s)] = counts_matrix

temporary fix for missing replicate-specific rna counts

In [ ]:
# temp fix for not having replicate-specific RNA counts matrix is to update the columns by merging with a transcript ID -> genome ID mapping
global_mapping = None
for pf in replicate_df[replicate_df['condition'] == 'TIS']['predict_file'].tolist():
    id_map = pd.read_csv(pf, sep='\t')[['Gid', 'Tid']].drop_duplicates()
    if global_mapping is None:
        global_mapping = id_map
    else:
        global_mapping = pd.concat([global_mapping, id_map], axis=0).drop_duplicates()
global_mapping = global_mapping.reset_index(drop=True)

In [38]:
pairs_to_count_matrix = dict()
for _, r in tisdiff_manifest.iterrows():
    baseline_s = r['baseline']
    test_s = r['test']

    counts_file = r['export_output_file'].replace('export_counts', 'export_counts_old')
    rna_table_file = r['rnaseq_merged_counts']

    counts_matrix = pd.read_csv(counts_file, sep='\t')
    counts_matrix_target_names = counts_matrix.columns[-4:]
    counts_matrix['Tid'] = counts_matrix['TIS'].apply(lambda x: x.split('_')[0])
    counts_matrix = counts_matrix.drop(counts_matrix_target_names, axis=1)

    rna_matrix = pd.read_csv(rna_table_file, sep='\t').rename({'ID': 'Gid'}, axis=1).set_index('Gid')
    rna_matrix.columns = counts_matrix_target_names
    rna_matrix = rna_matrix.merge(global_mapping, left_index=True, right_on=['Gid'], how='left').drop(['Gid'], axis=1).dropna(subset=['Tid'])

    counts_matrix = counts_matrix.merge(rna_matrix, how='left').drop(['Tid'], axis=1)
    counts_matrix[counts_matrix_target_names] = counts_matrix[counts_matrix_target_names].fillna(0)

    pairs_to_count_matrix[(test_s, baseline_s)] = counts_matrix


# Run DESeq2

From [Bioconductor forums](https://support.bioconductor.org/p/61509/):

    DESeq2 testing ratio of ratios (RIP-Seq, CLIP-Seq, ribosomal profiling)

    Suppose we have two assays: Input and IP, and we have two conditions: A and B.

    Then using a design: '~ assay + condition + assay:condition', the interaction term 'assay:condition' represents the ratio of the ratios: (IP for B / Input for B) / (IP for A / Input for A)... which with some algebra you can see is equivalent to (IP for B / IP for A) / (Input for B / Input for A).

    Using a likelihood ratio test which removes the interaction term in the reduced model, we test whether the IP enrichment is different in condition B than in A:

    dds <- DESeqDataSet(se, design= ~ assay + condition + assay:condition)
    dds <- DESeq(dds, test="LRT", reduced= ~ assay + condition)
    results(dds)

~Michael Love, lab of authors of DESeq2

In this sense, the LRT (likelihood ratio test) hypothesis test is whether there is ANY difference in translation efficiency (TE) between conditions, performed by comparing a model with and without the ratio term between riboseq and RNA

With exactly two conditions, this is roughly equivalent, but non-directional, to the Wald Test comparing the magnitude of the interaction coefficient to 0 (i.e. no difference in TE). But with more conditions, this distinction becomes clearer.

Specifically, the Wald Test will give, for each pair of condtions, the magnitude of the difference in translational efficiency; whereas the LRT still tests whether there is ANY difference in translation efficiency between any conditions.

Both are perhaps valuable: Wald giving pairwise foldchanges; and LRT giving a conservative measure of any outlier in TE.

## Prototype running DESeq through rpy2

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

deseq2 = importr("DESeq2")
apeglm = importr("apeglm")

def generate_coldata_table(column_names, sep='__'):
    metadata_df = pd.DataFrame({
        'colname': column_names,
        'condition': [x.split(sep)[0] for x in column_names],
        'replicate': [x.split(sep)[1] for x in column_names],
        'assay': [x.split(sep)[2] for x in column_names]
    }).set_index('colname')
    return metadata_df

In [12]:
counts = pairs_to_count_matrix[('K562', 'HeLa')]
coldata = generate_coldata_table(counts.columns)

with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(counts)
    r_coldata = pandas2ri.py2rpy(coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_dds = deseq2.DESeq(r_dds, test='LRT', reduced=ro.Formula('~ assay + condition'))

r_res_df = ro.r("as.data.frame")(deseq2.results(r_dds))

with localconverter(ro.default_converter + pandas2ri.converter):
    res_df = ro.conversion.rpy2py(r_res_df)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [32]:
res_df.sort_values('padj')

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
ENST00000598814.1_60_chr19:18573300:+,2762.300598,-14.624727,1.447718,2130.917241,0.000000e+00,0.000000e+00
ENST00000442744.7_61_chr19:18573300:+,2762.300598,-14.624727,1.447718,2130.917241,0.000000e+00,0.000000e+00
ENST00000650309.2_86_chrX:77899553:+,2082.100375,-12.803409,1.451732,908.363818,1.491545e-199,6.101166e-195
ENST00000318911.5_108_chr8:144095183:+,1689.806984,-7.114858,0.409883,823.363707,4.492221e-181,1.378157e-176
ENST00000579128.1_78_chr18:657742:+,1747.906223,-6.499005,0.258352,796.844572,2.618845e-175,4.591022e-171
...,...,...,...,...,...,...
ENST00000528931.5_2259_chr11:12256829:+,40.227384,-2.328665,1.791850,-0.007685,1.000000e+00,1.000000e+00
ENST00000487659.1_51_chr1:1407217:-,375.112090,0.006594,0.197048,0.001119,9.733089e-01,1.000000e+00
ENST00000482352.1_73_chr1:1407217:-,375.112090,0.006594,0.197048,0.001119,9.733089e-01,1.000000e+00
ENST00000537760.5_146_chr11:14295825:-,10.521111,-0.025405,1.710039,-0.000278,1.000000e+00,1.000000e+00


## Two questions can be answered with DESeq2

1) For any arbitrary pair of cell lines to compare, what is the difference in TE usage across cell lines/conditions?
    - Fit the global RNA counts and RPF counts matrix with 3 terms: the RNA counts, the RPF counts, and the interaction between the RNA and RPF counts: `~ assay + cell_line + assay:cell_line`
    - The coefficients of the interaction term should capture the differences in translational efficiency between cell lines
    - All pairwise LFCs (in translational efficiency) can be extracted with the `contrast` parameter for the interaction terms
    - A conservative measure of whether TE usage differs in any cell line can be tested with the `LRT` test

2) For a correlative analysis across all cell lines, how do the fitted means vary across cell lines?
    - Fit the global RNA counts and RPF counts matrix with 3 terms again
    - The means per sample can be extracted and compared across cell lines
    - A variance-stabilizing transformation (`vst()`) may be helpful so make low-count sites comparable to high-count sites, which according to the statistical model of NB-distributed counts, higher average counts necessarily lead to higher variances (mean-variance relationship; exacerbated by overdispersion)

## Global DESeq2 fit over all conditions

In [33]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

deseq2 = importr("DESeq2")

def generate_coldata_table(column_names, sep='__'):
    metadata_df = pd.DataFrame({
        'colname': column_names,
        'condition': [x.split(sep)[0] for x in column_names],
        'replicate': [x.split(sep)[1] for x in column_names],
        'assay': [x.split(sep)[2] for x in column_names]
    }).set_index('colname')
    return metadata_df

In [34]:
counts_by_sample = dict()
for pair, counts in pairs_to_count_matrix.items():
    s1, s2 = pair
    if s1 not in counts_by_sample:
        counts_by_sample[s1] = counts.loc[:, [c for c in counts.columns if s1 in c]]
    if s2 not in counts_by_sample:
        counts_by_sample[s2] = counts.loc[:, [c for c in counts.columns if s2 in c]]

In [35]:
all_counts = pd.concat(list(counts_by_sample.values()), axis=1).fillna(0)
all_counts = all_counts.astype(int)
coldata = generate_coldata_table(all_counts.columns)

## Run the LRT test for any difference in TE

In [37]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(all_counts)
    r_coldata = pandas2ri.py2rpy(coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_lrt_dds = deseq2.DESeq(r_dds, test='LRT', reduced=ro.Formula('~ assay + condition'))
r_lrt_res_df = ro.r("as.data.frame")(deseq2.results(r_lrt_dds))

with localconverter(ro.default_converter + pandas2ri.converter):
    lrt_res_df = ro.conversion.rpy2py(r_lrt_res_df)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [46]:
lrt_res_df.sort_values('padj').to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/lrt_results.csv')

## Run all pairwise Wald tests for relative LFCs

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(all_counts)
    r_coldata = pandas2ri.py2rpy(coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_wald_dds = deseq2.DESeq(r_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [ ]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(colData$condition))')[0]

In [ ]:
# for all pairwise comparisons, extract the results
pairwise_wald_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 == reference_sample:
        # swap the variables
        temp = s1
        s1 = s2
        s2 = temp
    if s2 == reference_sample:
        coeff_name = f'assayrpf.condition{s1}'
        res = ro.r(f'as.data.frame(results(r_wald_dds, name="{coeff_name}"))')
    else:
        contrast = f'list("assayrpf.condition{s1}", "assayrpf.condition{s2}")'
        res = ro.r(f'as.data.frame(results(r_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        pair_res_df = pandas2ri.rpy2py(res)

    pairwise_wald_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [02:01<00:00,  8.09s/it]


In [126]:
for p, df in tqdm(pairwise_wald_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/pairwise_wald/{s1}_vs_{s2}.csv')

100%|██████████| 15/15 [00:43<00:00,  2.92s/it]


## Extract the variance-stabilized transform matrix of mean TE per condition

In [ ]:
# run the transformation
r_vst = deseq2.vst(r_dds, blind=True)

# extract the transformed matrix
r_vst_mtx = ro.r('assay')(r_vst)
with localconverter(ro.default_converter + pandas2ri.converter):
    vst_mtx = pd.DataFrame(ro.conversion.rpy2py(r_vst_mtx), index=all_counts.index, columns=all_counts.columns)

# both RNA and Riboseq reads have been variance stabilized and logged (i.e. adjusted for mean-variance relationship), average over replicates per sample
vst_avg_mtx = vst_mtx.T.groupby([coldata['condition'], coldata['assay']]).mean().T

# subtracting log(RPF) - log(RNA) gives log(RPF/RNA) ~= log(TE)
vst_avg_te = vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rpf']].droplevel(1, axis=1) - vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rna']].droplevel(1, axis=1)

vst_avg_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/translation_efficiency_vst_matrix.csv')

In [ ]:
# shrunked_lfcs = deseq2.lfcShrink(r_wald_dds, coef=f'assayrpf.condition{s2}', type="apeglm")

R[write to console]: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



<rpy2.robjects.methods.RS4 object at 0x7f7aa1302f50> [25]
R classes: ('DESeqResults',)